# Consumindo uma API REST

Neste exemplo usamos a API REST [https://www.geojs.io/](https://www.geojs.io/).

Ela é simples, não tem limite de requisições e não exige autenticação — assim focamos na parte do Pydantic.

In [1]:
import json
from pathlib import Path

import requests
from pydantic import BaseModel, ConfigDict, Field, field_validator, ValidationError, IPvAnyAddress

> 📝 **Nota de revisão — execução offline:** este notebook foi reexecutado num ambiente sem acesso à geojs.io. Para continuar executável
> em qualquer lugar, a função `get_json` abaixo tenta a chamada real e, se a rede falhar, usa as respostas gravadas na saída do
> notebook original do curso (outubro de 2024), salvas em [`geojs_samples.json`](geojs_samples.json). A saída indica qual fonte foi usada.
> A resposta gravada de `23.62.177.155` é parcial (o curso só exibiu os campos do modelo).

In [2]:
SAMPLES = json.loads(Path("geojs_samples.json").read_text())


def get_json(url: str, ip_address: str) -> dict:
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        print(f"fonte: ao vivo ({url})")
        return response.json()
    except requests.RequestException as ex:
        print(f"fonte: amostra gravada (rede indisponível: {type(ex).__name__})")
        return SAMPLES[ip_address]

Normalmente, quando usamos modelos Pydantic para ler dados devolvidos por uma API, não precisamos de muita validação — a expectativa
é que a API seja consistente e devolva o que promete.

Claro, isso varia: dependendo da API, é preciso mais cuidado ao definir os modelos. E sim, infelizmente há muitas APIs mal projetadas por aí!
Você pode acabar escrevendo muitos validadores customizados só para tornar os dados recebidos consistentes e fáceis de usar na aplicação.

## Endpoint

Vamos usar os endpoints `Geo` desta API. Dado um endereço IP (ou nenhum — aí ela usa o seu IP público), ela devolve informações
geográficas sobre o IP.

É uma requisição `GET`, então não precisamos modelar um payload JSON, só a resposta.

Infelizmente, eles não fornecem um JSON Schema (pelo menos o instrutor não encontrou), então criamos o modelo Pydantic do zero.

Como não nos interessam todos os campos devolvidos pelo endpoint, criamos um modelo com `extra="ignore"`.

Além disso, esta API não usa camel case nos nomes dos campos, então nem precisamos de aliases.

A documentação não diz se os campos da resposta podem ser nulos.

A abordagem do instrutor: tornar **obrigatórios** os campos de que o programa absolutamente precisa e **anuláveis** os demais,
tratando `None` no código.

A documentação diz que `organization_name` vem como a string `Unknown` quando o valor é desconhecido — o instrutor não gosta de valores
especiais para representar nulo, então usa um validador (*after*, para o Pydantic ao menos validar/coagir a string) que troca `Unknown`
por `None` (com o campo anulável, claro).

In [3]:
class IPGeo(BaseModel):
    model_config = ConfigDict(extra="ignore")

    ip: IPvAnyAddress
    country: str | None = None
    country_code: str | None = Field(default=None, min_length=2, max_length=2)
    country_code3: str | None = Field(default=None, min_length=2, max_length=3)
    city: str | None = None
    region: str | None = None
    timezone: str | None= None
    organization_name: str | None = None

    @field_validator("organization_name", mode="after")
    @classmethod
    def set_unknown_to_none(cls, value: str):
        if value.casefold() == "unknown":
            return None
        return value

In [4]:
print(f'IPGeo(ip="8.8.8.8", country="test", country_code3 = "USA", organization_name="Unknown"): {IPGeo(ip="8.8.8.8", country="test", country_code3 = "USA", organization_name="Unknown")!r}')

IPGeo(ip="8.8.8.8", country="test", country_code3 = "USA", organization_name="Unknown"): IPGeo(ip=IPv4Address('8.8.8.8'), country='test', country_code=None, country_code3='USA', city=None, region=None, timezone=None, organization_name=None)


> 📝 **Nota de revisão — bug:** o validador recebe `str | None` (é *after*, roda depois da validação do tipo `str | None`), mas assume
> `str`. Se a API mandar `"organization_name": null` — e a própria documentação não garante que não mande —, `None.casefold()` estoura.
> O erro vira um `AttributeError` cru, nem é um `ValidationError`. (Com o default, o validador nem roda, porque `validate_default` é `False`.)

In [5]:
try:
    IPGeo(ip="8.8.8.8", organization_name=None)
except Exception as ex:
    print(f"{type(ex).__name__}: {ex}")

AttributeError: 'NoneType' object has no attribute 'casefold'


Correção — tratar `None` antes:

In [6]:
class IPGeo(BaseModel):
    model_config = ConfigDict(extra="ignore")

    ip: IPvAnyAddress
    country: str | None = None
    country_code: str | None = Field(default=None, min_length=2, max_length=2)
    country_code3: str | None = Field(default=None, min_length=2, max_length=3)
    city: str | None = None
    region: str | None = None
    timezone: str | None = None
    organization_name: str | None = None

    @field_validator("organization_name", mode="after")
    @classmethod
    def set_unknown_to_none(cls, value: str | None) -> str | None:
        if value is None or value.casefold() == "unknown":
            return None
        return value

print(f"organization_name=None: {IPGeo(ip='8.8.8.8', organization_name=None).organization_name!r}")
print(f"organization_name='Unknown': {IPGeo(ip='8.8.8.8', organization_name='Unknown').organization_name!r}")

organization_name=None: None
organization_name='Unknown': None


Agora com o endpoint. Primeiro, especificando um IP:

In [7]:
url_query = "https://get.geojs.io/v1/geo/{ip_address}.json"

In [8]:
url = url_query.format(ip_address="8.8.8.8")
response_json = get_json(url, "8.8.8.8")

fonte: amostra gravada (rede indisponível: ProxyError)


In [9]:
print(f"response_json: {response_json}")

response_json: {'country': 'United States', 'country_code': 'US', 'country_code3': 'USA', 'continent_code': 'NA', 'asn': 15169, 'organization_name': 'GOOGLE', 'organization': 'AS15169 GOOGLE', 'timezone': 'America/Chicago', 'accuracy': 1000, 'ip': '8.8.8.8', 'longitude': '-97.822', 'latitude': '37.751', 'area_code': '0'}


Para desserializar: a biblioteca `requests` já converte a string JSON num dicionário Python (`response.json()`), então usamos
`model_validate()`, não `model_validate_json()`.

In [10]:
data = IPGeo.model_validate(response_json)

In [11]:
print(f"data: {data!r}")

data: IPGeo(ip=IPv4Address('8.8.8.8'), country='United States', country_code='US', country_code3='USA', city=None, region=None, timezone='America/Chicago', organization_name='GOOGLE')


Repare que os campos extras (`asn`, `latitude`, `longitude`, …) foram descartados e que `city`/`region`, ausentes nesta resposta,
ficaram `None` — exatamente por isso eles foram declarados como anuláveis.

Mais um:

In [12]:
url = url_query.format(ip_address="23.62.177.155")
data = IPGeo.model_validate(get_json(url, "23.62.177.155"))
print(f"data.model_dump_json(indent=2):\n{data.model_dump_json(indent=2)}")

fonte: amostra gravada (rede indisponível: ProxyError)
data.model_dump_json(indent=2):
{
  "ip": "23.62.177.155",
  "country": "United States",
  "country_code": "US",
  "country_code3": "USA",
  "city": "El Segundo",
  "region": "California",
  "timezone": "America/Los_Angeles",
  "organization_name": "AKAMAI-AS"
}


Também dá para consultar o próprio IP público (a GeoJS descobre sozinha):

```python
url = "https://get.geojs.io/v1/ip/geo.json"
response = requests.get(url, timeout=5)
response.raise_for_status()
data = IPGeo.model_validate(response.json())
print(data.model_dump_json(indent=2))
```

> 📝 **Nota de revisão:** no original, `response.raise_for_status` estava **sem parênteses** — só referencia o método, não verifica nada.
> Também vale sempre passar `timeout=` ao `requests.get` (sem ele, a chamada pode ficar pendurada indefinidamente).

O instrutor não executou esse trecho para não expor o próprio IP público :-)

> 💡 **Complemento — resposta fora do contrato:** o valor do modelo é justamente transformar uma resposta inesperada num erro **claro e
> localizado**, em vez de um `KeyError` ou `None` silencioso lá na frente:

In [13]:
try:
    IPGeo.model_validate({"ip": "999.1.1.1", "country_code": "USA"})
except ValidationError as ex:
    for error in ex.errors():
        print(f"{error['loc']}: {error['msg']}")

('ip',): value is not a valid IPv4 or IPv6 address
('country_code',): String should have at most 2 characters
